# Graph analysis of the Tokopedia seller network

The 120 listings in `laptop_listings_filled.json` were **scraped from
Tokopedia**. They are the only real-world data in this project — everything else
is synthetic — so a finding here is an actual finding rather than a restatement
of assumptions a generator was given.

Two questions, in order:

1. **Is the data what it claims to be?** It is not, quite. The scrape mixes
   accessories in with laptops, and the `is_suspected_scam` flag inherited from
   the pipeline turns out to be wrong on every single row it fires on.
2. **Does a graph find anything a `groupby` would not?** Partly. One of the three
   constructions below earns its place; the other two are reported honestly as
   not doing so, because a centrality score that ranks nodes identically to
   plain degree is decoration.

**On tooling.** This uses `networkx`, not Neo4j. At 51 sellers and 120 listings
every algorithm here runs in milliseconds in memory; a graph database would add
an infrastructure dependency without changing one result. Neo4j and its GDS
library earn their keep at a scale this dataset is several orders of magnitude
below, and saying so is a better answer than installing it for appearance.

## 1. Setup

In [ ]:
import collections
import itertools
import json

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

import graph_analysis as ga

listings = ga.load_listings()
print(f"{len(listings)} scraped listings")
print("marketplaces:", collections.Counter(x["source"] for x in listings))
print("scraped at  :", listings[0]["scraped_at"][:10])
pd.DataFrame(listings).head(3)[["title", "brand", "model", "price_idr", "seller_name", "location"]]

## 2. Data quality audit

Before any algorithm runs. Graph results inherit every defect in the node set,
and this node set has two.

In [ ]:
audit = ga.audit_scam_flags(listings)
for key, value in audit.items():
    print(f"  {key:32} {value}")

### Finding 1 — the scam detector is wrong on every row it fires on

`flagged_that_are_laptops` is **0**. All 26 listings marked
`is_suspected_scam` are accessories — keyboards, chargers, stands, a backpack,
two "bonus" promo entries — caught by a single rule,
`price_far_below_market(<3500000)`, which is the entire reason vocabulary.

A keyboard at Rp 123,750 is not a suspiciously cheap laptop. It is a keyboard.
The rule has no notion of *what* is being priced, so it fires on every cheap
item in a dataset that turned out to contain many.

This matters beyond tidiness: `is_suspected_scam` is indexed into Qdrant as
listing metadata and is a hard-floor condition in the sourcing agent. A detector
with a 100% false-positive rate that also finds nothing real is worse than none,
because it looks like coverage.

In [ ]:
tagged = ga.classify_listings(listings)
flagged = [x for x in tagged if x["is_suspected_scam"]]

print("Every listing the current rule flags:\n")
for x in sorted(flagged, key=lambda r: r["price_idr"])[:12]:
    print(f"  {x['price_idr']:>12,}  {x['title'][:66]}")
print(f"  ... {len(flagged) - 12} more, all of the same kind")

laptops = ga.laptops_only(listings)
print(f"\nAfter cleaning: {len(laptops)} laptops, "
      f"{len(listings) - len(laptops)} accessories removed.")
prices = sorted(x["price_idr"] for x in laptops)
print(f"Laptop price range: {prices[0]:,} - {prices[-1]:,} IDR "
      f"(median {prices[len(prices) // 2]:,})")

### Finding 2 — the scrape is one brand on one marketplace

Every listing is Asus, and every listing is Tokopedia. That bounds what any
conclusion below can claim: this is the structure of *the Asus reseller network
on one platform*, not of Indonesian laptop retail. Worth stating plainly rather
than letting the reader assume breadth the sample does not have.

In [ ]:
print("brands     :", collections.Counter(x["brand"] for x in listings))
print("sellers    :", len({x["seller_name"] for x in listings}))
print("models     :", len({x.get("model") for x in listings}))
print("locations  :", dict(collections.Counter(x["location"] for x in listings).most_common(6)))

## 3. Three graph constructions

| Graph | Edge means | Verdict |
|---|---|---|
| Seller–Model bipartite | this seller lists this model | centrality adds nothing (§4) |
| Seller co-listing | we both sell the same model | weak; popularity, not relationship (§5) |
| Seller title-similarity | our listing text is near-identical | **the one that works** (§6) |

In [ ]:
B = ga.build_bipartite(listings)
P = ga.build_colisting_projection(listings)
F = ga.build_family_graph(listings)

for name, G in [("bipartite", B), ("co-listing", P), ("title-similarity", F)]:
    print(f"  {name:18} {G.number_of_nodes():3} nodes, {G.number_of_edges():4} edges, "
          f"density {nx.density(G):.3f}")

## 4. Does centrality earn its place? (No.)

The standard move is to run PageRank or betweenness and report the top node as
an "influential entity". The check nobody runs is whether that ranking differs
from plain **degree** — because if it does not, `value_counts()` would have
produced the same answer and the graph was ceremony.

In [ ]:
check = ga.centrality_vs_degree(B, kind="seller")
print(f"Spearman rho (PageRank vs degree): {check['spearman_rho']:.3f}")
print(f"identical top-5 set             : {check['same_top5']}")
print(f"\n  by PageRank: {check['top5_pagerank']}")
print(f"  by degree  : {check['top5_degree']}")

ρ = 0.82 and **the top-5 sets are identical** — only the internal order shifts.
On a graph this shallow, PageRank is a smoothed degree count.

So the honest conclusion is a negative one: *there is no influential-entity
finding here that counting listings would not have given you.* The sellers with
the most listings are the sellers with the most listings. Reporting that as a
centrality result would dress a `value_counts()` in graph vocabulary.

Betweenness would fare no better and for a sharper reason: in a bipartite
seller–model graph, every path between two sellers runs through a model node, so
betweenness measures how many popular models a seller stocks. That is, again,
degree.

## 5. Co-listing communities — weak, and it should be

The obvious projection: link two sellers when they list the same model. Run
Louvain and read off "vendor communities."

The problem is what the edge means. Thirteen sellers listing a Vivobook 14 are
not a community; a Vivobook 14 is a popular laptop. The edge encodes catalogue
overlap, and in a market where most sellers stock the same few models, that is
close to a complete graph carrying no information.

In [ ]:
communities, modularity = ga.detect_families(P)
print(f"co-listing graph: {P.number_of_nodes()} sellers, {P.number_of_edges()} edges")
print(f"Louvain communities: {[len(c) for c in communities]}")
print(f"modularity: {modularity:.3f}")
print("\nModularity below ~0.3 means barely more structure than a random graph "
      "of the same degree sequence.")

## 6. Seller families from shared listing text — the finding

Change what the edge means. Link two sellers when their **listing titles are
near-duplicates** — a copied product description, the same distributor feed, or
one operation running several storefronts.

Two properties make this worth a graph rather than a join:

- **It is transitive.** If A shares text with B and B with C, then {A, B, C} is a
  cluster even when A and C never overlap directly. No `groupby` returns that;
  it is a connected-components question.
- **Seller names never enter the comparison.** Only title text is compared, so
  any same-business pair the method recovers is an independent discovery rather
  than string-matching on names.

In [ ]:
families, modularity = ga.detect_families(F)
weights, best = ga.title_similarity_pairs(listings)

print(f"threshold: {ga.TITLE_SIMILARITY_THRESHOLD}  (chosen by sweep -- looser merges "
      f"every seller into one blob, tighter splits known storefronts apart)")
print(f"{F.number_of_nodes()} sellers connected by {F.number_of_edges()} edges")
print(f"Louvain modularity: {modularity:.3f}\n")

for i, members in enumerate(families, 1):
    if len(members) < 2:
        continue
    info = ga.describe_family(listings, members, weights)
    print(f"family {i} -- {info['size']} storefronts, {info['listings']} listings")
    for m in info["members"]:
        print(f"    {m}")
    print(f"    locations: {info['locations']}")
    if info["median_price_gap"] is not None:
        print(f"    median price gap between members: {info['median_price_gap']:.1%}")
    print()

### The validation: "Agres ID"

The method, which never compares seller names, places **Agres ID Electronics**
and **Agres ID Surabaya** in the same family — and at a looser threshold pulls in
Agres ID Bintaro, Agres ID Jakarta Utara and Agres ID Tangerang too. Those are
transparently one business running storefronts under one banner, and the graph
found them from product text alone.

That is the sanity check the technique needed. Having recovered a grouping we
can verify by eye, the *other* members of those families — sellers whose names
give nothing away — become credible rather than speculative.

**What the finding is not.** A shared distributor feed is ordinary retail, not
fraud. The actionable statement is narrower and still useful: *these quotes are
not independent.* A procurement officer comparing three prices from three
storefronts in one family is comparing one supplier's price three times.

In [ ]:
# Draw the family graph. Node size = listings, edge width = matched pairs.
fig, ax = plt.subplots(figsize=(13, 8))
pos = nx.spring_layout(F, seed=42, k=0.85)

counts = collections.Counter(x["seller_name"] for x in listings)
palette = plt.cm.tab10.colors
member_of = {m: i for i, fam in enumerate(families) for m in fam}

nx.draw_networkx_edges(
    F, pos, ax=ax, alpha=0.45,
    width=[0.7 + 0.5 * F[u][v]["weight"] for u, v in F.edges()],
)
nx.draw_networkx_nodes(
    F, pos, ax=ax,
    node_size=[160 + 90 * counts[n] for n in F.nodes()],
    node_color=[palette[member_of.get(n, 9) % 10] for n in F.nodes()],
    edgecolors="white", linewidths=1.4,
)
nx.draw_networkx_labels(F, pos, ax=ax, font_size=8)
ax.set_title("Sellers linked by near-duplicate listing text (colour = Louvain family)")
ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Price dispersion — and calling it what it is

The other half of anomaly detection is comparing what different sellers charge
for the same machine. This is a **`groupby`**, not a graph algorithm, and it is
labelled as one here rather than folded into the network results to make them
look richer.

It matters because it is where the two halves meet: a wide spread is only
suspicious if the quotes are independent, and §6 is what tells you whether
they are.

In [ ]:
spread = pd.DataFrame(ga.price_dispersion(laptops))
spread["min_price"] = spread.min_price.map("{:,.0f}".format)
spread["max_price"] = spread.max_price.map("{:,.0f}".format)
spread["spread_pct"] = spread.spread_pct.map("{:.0%}".format)
spread.head(8)

A 42% spread on the Vivobook 14 across 13 sellers is the kind of gap the
procurement assistant should surface — the same machine at Rp 9.4M and at
Rp 16.1M. Whether that is a bargain or a bait listing depends on who is quoting
it, which is exactly what the family graph answers.

## 8. Feeding it back into the assistant

The graph result is exported as a seller → family map, and `SourcingAgent`
accepts it and emits a **disclosure** for any presented option whose seller sits
in a family. A disclosure, not a floor: sharing a distributor feed is ordinary
retail, and the agent's hard floor is reserved for things that make a
recommendation wrong.

**Be precise about what is live.** The agent currently sources candidates from
the *catalogue*, whose rows carry no seller, so the hook produces no warnings
today. It is loaded and correct, and it starts firing the moment marketplace
listings become candidates — which is the next change to make, not one already
made.

The second use is more immediately actionable: **retire the broken scam rule**.
`price_far_below_market` fires on accessories and catches nothing else. A floor
conditioned on the *model* rather than a global constant, combined with family
membership, is a defensible replacement for a detector that is currently 100%
false positives.

In [ ]:
payload = ga.export_seller_families(listings)
print(f"exported {len(payload['families'])} families "
      f"(modularity {payload['modularity']:.3f}) -> models/seller_families.json")

print("\nWhat the agent would say:\n")
for seller in ["Agres ID Surabaya", "Gateway Indonesia Comp", "House Of IT", "hans computer"]:
    warning = ga.family_warning(payload, seller)
    print(f"  {seller:26} {warning or '(no family -- quotes are independent)'}")

## 9. What this does and does not establish

**Established.** The inherited scam flag is wrong on all 26 rows it fires on and
should be replaced. Five seller families exist in the Asus/Tokopedia sample,
recovered from listing text without reference to seller names, and validated
against a grouping (Agres ID) that is verifiable by eye. Quotes within a family
are not independent, which changes how a price comparison should be read.

**Not established.** Nothing about fraud — a shared distributor feed is normal
retail. Nothing about vendor concentration in *BNI's own procurement*, because
the procurement records carry `vendor_risk_score` and `vendor_is_official` but no
vendor **identity**; there is no division→vendor edge to build, and inventing one
would only rediscover the generator's assumptions. Nothing beyond Asus on
Tokopedia.

**What would raise the ceiling.** Seller registration numbers or bank accounts
would turn "shared listing text" into genuine identity resolution. A second
marketplace would make the `SELLS_ON` relation carry information instead of
pointing at one constant. And real procurement records with vendor names would
finally make the division↔vendor graph — the most-requested analysis — something
other than an echo of synthetic data.

**On scale, honestly.** 51 sellers is small. Louvain at modularity 0.43 is
meaningful but not strong, and it would be overreach to present these families as
anything more than a lead worth a human check. The technique is sound and would
sharpen considerably on a larger scrape; the finding it produced today is a
data-quality bug and a handful of storefront groupings — which is a real result,
and a smaller one than the method could deliver with more data.